In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install openai -q

In [ ]:
import json, os, re, time, subprocess, ast
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

INPUT_DIR = '/kaggle/input/datasets/apr-dataset'
OUTPUT_DIR = '/kaggle/working'

KB_PATH = f'{INPUT_DIR}/pattern_knowledge_base_final.json'
TEST_PATH = f'{INPUT_DIR}/test_loop_filtered_verified.jsonl'
TEST_CASES_DIR = f'{INPUT_DIR}/public_test_cases/codenet/public_test_cases'
OUTPUT_PATH = f'{OUTPUT_DIR}/fasterpy_results_gpt54mini_k10.jsonl'

SPEEDUP_THRESHOLD = 1.10
MAX_ATTEMPTS = 10
GPT_MODEL = "gpt-5.4-mini"  # verify exact string in OpenAI's docs before running

user_secrets = UserSecretsClient()
client = OpenAI(api_key=user_secrets.get_secret("OPENAI_API_KEY"))

with open(KB_PATH) as f:
    pattern_knowledge_base = json.load(f)
print(f'Loaded KB: {len(pattern_knowledge_base)} patterns')

test_samples = []
with open(TEST_PATH) as f:
    for line in f:
        test_samples.append(json.loads(line))
print(f'Loaded {len(test_samples)} test samples')

print('Test cases dir exists:', os.path.isdir(TEST_CASES_DIR))

In [ ]:
!git clone https://github.com/WuYue22/FasterPy.git
%cd FasterPy

In [ ]:
!pip install -r requirements.txt
!pip install pymilvus "pymilvus[milvus_lite]"
!pip install -U bitsandbytes accelerate transformers -q

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="microsoft/unixcoder-base", local_dir="./unixcoder-base")

In [ ]:
from knowledge.knowledge_base import KnowledgeBase
kb = KnowledgeBase(db_name="CKB")

stats = kb.client.get_collection_stats(collection_name="demo_collection")
print("Before insertion:", stats)  # should be 0 here, that's expected/fine at this point

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import json
from tqdm import tqdm

with open('/kaggle/input/datasets/apr-dataset/kb_summaries_checkpoint (1).jsonl') as f:
    summaries = [json.loads(line) for line in f]
print(f'{len(summaries)} summaries loaded')

def prep_record(rec, max_len=290):
    summary = rec["summary"][:max_len]
    return {
        "src_code": rec["input"],
        "src_code_len": len(rec["input"].splitlines()),
        "summary": summary,
        "rate": rec["speedup_ratio"],
    }

insert_records = [prep_record(rec) for rec in summaries]

BATCH_SIZE = 500
for i in tqdm(range(0, len(insert_records), BATCH_SIZE)):
    kb.insert(insert_records[i:i + BATCH_SIZE])

print("Insertion complete")

In [ ]:
stats = kb.client.get_collection_stats(collection_name="demo_collection")
print("After insertion:", stats)
# row_count ekhon shob summary-r shomoperiman (~25,092) howa uchit, 0 NA
assert stats['row_count'] > 20000, "KB still empty or partially inserted — DO NOT proceed to retrieval yet!"

In [ ]:
import pandas as pd
from tqdm import tqdm

INPUT_DIR = '/kaggle/input/datasets/apr-dataset'  # tomar actual input-path
OUTPUT_DIR = '/kaggle/working'

df = pd.read_json(f"{INPUT_DIR}/test_loop_filtered_verified.jsonl", lines=True)

code_col = "slow_code_col" if "slow_code_col" in df.columns else "input"

suggestions_list = []
for code in tqdm(df[code_col]):
    result = kb.search([code], top_k=3)
    suggestions = [{"text": hit["entity"]["summary"], "rate": hit["entity"]["rate"], "distance": hit["distance"]} for hit in result[0]]
    suggestions_list.append(suggestions)

df["suggestion"] = suggestions_list
if code_col != "slow_code_col":
    df["slow_code_col"] = df[code_col]

df.to_json(f"{OUTPUT_DIR}/test_with_suggestions.jsonl", lines=True, orient="records")
print(f"Saved to {OUTPUT_DIR}/test_with_suggestions.jsonl")

print(df["suggestion"].iloc[0])

In [ ]:
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
import time

GPT_MODEL = "gpt-5.4-mini"  # verify exact model string in OpenAI's docs before running

user_secrets = UserSecretsClient()
client = OpenAI(api_key=user_secrets.get_secret("OPENAI_API_KEY"))

def call_gen_llm(prompt, max_new_tokens=1024, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.15,
                max_completion_tokens=max_new_tokens,
            )
            content = response.choices[0].message.content
            return content.strip() if content else ''
        except Exception as e:
            err_str = str(e).lower()
            if 'rate_limit' in err_str or '429' in err_str:
                wait_time = 15 * (attempt + 1)
                print(f'   Rate limited, waiting {wait_time}s...')
                time.sleep(wait_time)
            else:
                print(f'   API error: {e}')
                return ''
    return ''

print(f'{GPT_MODEL} ready (temperature=0.3, max_new_tokens=1024).')

In [ ]:
import re
from string import Template
from pygments.lexers import guess_lexer
from pygments.util import ClassNotFound
from tqdm import tqdm

def is_probably_python(text):
    try:
        return 'Python' in guess_lexer(text).name
    except ClassNotFound:
        return False

def extract_code(text):
    m = re.findall(r"```python\s*(.*?)```", text, re.DOTALL)
    if m: return m[0].strip()
    m = re.findall(r"```\s*(.*?)```", text, re.DOTALL)
    if m: return m[0].strip()
    parts = text.split("```")
    if is_probably_python(parts[0]): return parts[0].strip()
    if is_probably_python(parts[-1]): return parts[-1].strip()
    return text.strip()

input_template = Template(
    "Input code:\n$input\nSuggestions:\n$suggestions\nOutput: ```python\n {{optimized code}} \n``` <|EOS|>"
)
system = """You are an AI programming assistant. Try your best to optimize the Input code with focusing on reducing its runtime while maintaining correctness.
Feel free to refer to the suggestions below for potential optimizations, but you're not restricted to them.
Applicability represents the degree to which a suggestion fits the input code.
Rate represents the degree to which a suggestion improves the input code."""

def infer_row_gpt(row, max_retries=5):
    suggestions = row["suggestion"]
    sugg_text = "\n".join(
        f"{i+1}. Applicability:{s['distance']} Rate:{s['rate']}\n{s['text']}"
        for i, s in enumerate(suggestions)
    ) if suggestions else "None."
    prompt = input_template.substitute(input=row["slow_code_col"], suggestions=sugg_text)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
                temperature=0,
                max_completion_tokens=512,
            )
            content = response.choices[0].message.content
            return extract_code(content.strip() if content else '')
        except Exception as e:
            err_str = str(e).lower()
            if 'rate_limit' in err_str or '429' in err_str:
                wait_time = 15 * (attempt + 1)
                print(f'   Rate limited, waiting {wait_time}s...')
                time.sleep(wait_time)
            else:
                print(f'   API error: {e}')
                return ''
    return ''

tqdm.pandas()
df["model_generated_potentially_faster_code_col"] = df.progress_apply(infer_row_gpt, axis=1)
df.to_json(f"{OUTPUT_DIR}/fasterpy_baseline_generated.jsonl", lines=True, orient="records")
print(df.shape)
print(df["model_generated_potentially_faster_code_col"].iloc[0][:300])

In [ ]:
TEST_CASES_DIR = f'{INPUT_DIR}/public_test_cases/codenet/public_test_cases'
print("Test cases dir exists:", os.path.isdir(TEST_CASES_DIR))

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/working'):
    if 'FasterPy' in root:
        print(root)
        break
else:
    print('FasterPy not found in /kaggle/working')

print(os.getcwd())

In [ ]:
%%writefile /kaggle/working/FasterPy/eval/sample_eval_config.yaml
model_generated_outputs_path: "/kaggle/working/fasterpy_baseline_generated.jsonl"
inputs_outputs_basepath: "/kaggle/input/datasets/apr-dataset/public_test_cases/codenet/public_test_cases"
reference_file_path: "eval/py_reference.jsonl"
output_report_file_path: "/kaggle/working/fasterpy_baseline_eval_report.jsonl"
language: "python"
num_problems_to_evaluate: -1
num_trials: 4
ignore_first_k: 1
max_time_per_run: 3
temp_dir: null
model_generated_potentially_faster_code_col: "model_generated_potentially_faster_code_col"
slow_code_col: "slow_code_col"
reference_code_col: "target"
is_prompt_based: false
cpu_number: 1

In [ ]:
print(os.path.exists('/kaggle/working/FasterPy/eval/py_reference.jsonl'))

In [ ]:
!pip install cirron
!pip install git+https://github.com/s7nfo/Cirron.git

In [ ]:
import os
old_report = '/kaggle/working/fasterpy_baseline_eval_report.jsonl'
if os.path.exists(old_report):
    os.remove(old_report)
    print('Old report removed -- ensuring fresh output.')
else:
    print('No old report found -- clean start.')

In [ ]:
!cd /kaggle/working/FasterPy/eval/src/codenet_eval && python run_eval.py --eval_config /kaggle/working/FasterPy/eval/sample_eval_config.yaml

In [ ]:
import json

report = []
with open('/kaggle/working/fasterpy_baseline_eval_report.jsonl') as f:
    for line in f:
        report.append(json.loads(line))

print(f'Total evaluated: {len(report)}')
print(list(report[0].keys()))

In [ ]:
import pandas as pd

report_df = pd.DataFrame(report)

acc_col = 'model_generated_potentially_faster_code_col_acc'
time_col = 'model_generated_potentially_faster_code_col_time_mean'

# Official Pass@1 -- correctness only
pass1 = report_df[acc_col].mean() * 100

# Official %OPT -- speedup > 10% AND correct
faster_count = 0
total = len(report_df)
for _, row in report_df.iterrows():
    if row[acc_col] != 1:
        continue
    baseline_t = row['input_time_mean']
    model_t = row[time_col]
    if model_t is None or model_t <= 0 or baseline_t is None:
        continue
    speedup = (baseline_t / model_t) - 1
    if speedup > 0.1:
        faster_count += 1

opt_pct = 100 * faster_count / total

print(f'Total: {total}')
print(f'Pass@1 (official): {pass1:.1f}%')
print(f'%OPT (official, speedup>10% AND correct): {opt_pct:.1f}%')

In [ ]:
ratios = (report_df['input_time_mean'] / report_df[time_col]).replace([float('inf'), -float('inf')], None).dropna()
print(f'Median speedup: {ratios.median():.3f}x')
print(f'Mean speedup: {ratios.mean():.3f}x')

In [ ]:
# Step 1: Check the raw generated file (before evaluation) for duplicates
import json

gen_rows = []
with open('/kaggle/working/fasterpy_baseline_generated.jsonl') as f:
    for line in f:
        gen_rows.append(json.loads(line))

print(f'Step 1 - Raw generated file: {len(gen_rows)} rows')
print(f'Unique problem_ids in generated file: {len(set(r["problem_id"] for r in gen_rows))}')

In [ ]:
import subprocess, time

SPEEDUP_THRESHOLD = 1.10

def load_test_cases(problem_id, test_cases_dir):
    problem_dir = os.path.join(test_cases_dir, problem_id)
    if not os.path.isdir(problem_dir):
        return []
    cases = []
    i = 0
    while True:
        input_path = os.path.join(problem_dir, f'input.{i}.txt')
        output_path = os.path.join(problem_dir, f'output.{i}.txt')
        if not os.path.exists(input_path):
            break
        with open(input_path) as f:
            stdin_data = f.read()
        with open(output_path) as f:
            expected_output = f.read().strip()
        cases.append({'stdin': stdin_data, 'expected': expected_output})
        i += 1
    return cases

def check_compiles(code):
    try:
        compile(code, '<repair>', 'exec')
        return True, None
    except SyntaxError as e:
        return False, str(e)

def run_with_timeout(code, stdin_data, timeout_sec=5):
    start = time.time()
    try:
        result = subprocess.run(['python3', '-c', code], input=stdin_data,
                                 capture_output=True, text=True, timeout=timeout_sec)
        elapsed = time.time() - start
        return result.stdout.strip(), result.returncode == 0, elapsed
    except subprocess.TimeoutExpired:
        return None, False, timeout_sec

def check_functional_correctness(code, test_cases):
    for case in test_cases:
        output, success, _ = run_with_timeout(code, case['stdin'])
        if not success or output != case['expected']:
            return False, {**case, 'actual_output': output if output is not None else '(crashed or timed out)'}
    return True, None

def measure_runtime(code, test_cases, repeats=3):
    total_time = 0
    for _ in range(repeats):
        for case in test_cases:
            _, success, elapsed = run_with_timeout(code, case['stdin'])
            if not success:
                return None
            total_time += elapsed
    return total_time / repeats

def validate_repair(slow_code, repaired_code, problem_id, test_cases_dir, speedup_threshold=SPEEDUP_THRESHOLD):
    test_cases = load_test_cases(problem_id, test_cases_dir)
    if not test_cases:
        return {'verdict': 'RETRY', 'reason': f'No test cases found for {problem_id}'}
    compiles, syntax_error = check_compiles(repaired_code)
    if not compiles:
        return {'verdict': 'RETRY', 'reason': 'compile_error', 'detail': syntax_error}
    correct, failing_case = check_functional_correctness(repaired_code, test_cases)
    if not correct:
        return {'verdict': 'RETRY', 'reason': 'test_failure', 'detail': failing_case}
    slow_runtime = measure_runtime(slow_code, test_cases)
    repaired_runtime = measure_runtime(repaired_code, test_cases)
    if slow_runtime is None or repaired_runtime is None or repaired_runtime <= 0:
        return {'verdict': 'RETRY', 'reason': 'runtime_measurement_failed'}
    speedup = slow_runtime / repaired_runtime
    if speedup >= speedup_threshold:
        return {'verdict': 'ACCEPT', 'speedup': round(speedup, 3),
                'slow_runtime': round(slow_runtime, 4), 'repaired_runtime': round(repaired_runtime, 4)}
    else:
        return {'verdict': 'RETRY', 'reason': 'insufficient_speedup', 'speedup': round(speedup, 3),
                'slow_runtime': round(slow_runtime, 4), 'repaired_runtime': round(repaired_runtime, 4)}

print('validate_repair ready.')

In [ ]:
print(TEST_CASES_DIR)

In [ ]:
my_verdict = validate_repair(slow_code, fasterpy_code, sample_pid, TEST_CASES_DIR)
print(f'Your own validate_repair verdict: {my_verdict}')

In [ ]:
import random
random.seed(1)
sample_pids = random.sample(list(report_df[report_df[acc_col] == 1]['problem_id']), min(10, sum(report_df[acc_col]==1)))

agree_count = 0
disagree_count = 0

for pid in sample_pids:
    row = report_df[report_df['problem_id'] == pid].iloc[0]
    fpy_code = row['model_generated_potentially_faster_code_col']
    slow = row['slow_code_col']

    my_result = validate_repair(slow, fpy_code, pid, TEST_CASES_DIR)
    fasterpy_verdict = 'ACCEPT' if row[acc_col] == 1 else 'REJECT'
    my_verdict_simple = 'ACCEPT' if my_result['verdict'] == 'ACCEPT' else 'REJECT'

    match = "MATCH" if fasterpy_verdict == my_verdict_simple else "MISMATCH"
    if match == "MATCH":
        agree_count += 1
    else:
        disagree_count += 1
    print(f'{pid}: FasterPy-eval={fasterpy_verdict}, Your-eval={my_verdict_simple} [{match}]')

print(f'\nAgreement: {agree_count}/{len(sample_pids)}')

In [ ]:
row = report_df[report_df['problem_id'] == 'p03208'].iloc[0]
fpy_code = row['model_generated_potentially_faster_code_col']
slow = row['slow_code_col']

my_result = validate_repair(slow, fpy_code, 'p03208', TEST_CASES_DIR)
print('Your verdict:', my_result)
print()
print('Generated code:')
print(fpy_code[:500])

In [ ]:
fasterpy_rescored = []

for _, row in report_df.iterrows():
    problem_id = row['problem_id']
    slow_code = row['slow_code_col']
    generated_code = row['model_generated_potentially_faster_code_col']

    if not generated_code or not generated_code.strip():
        fasterpy_rescored.append({'problem_id': problem_id, 'verdict': 'REFUSED_OR_EMPTY'})
        continue

    result = validate_repair(slow_code, generated_code, problem_id, TEST_CASES_DIR)
    fasterpy_rescored.append({'problem_id': problem_id, **result})

print(f'Total re-scored: {len(fasterpy_rescored)}')

In [ ]:
accepted = [r for r in fasterpy_rescored if r.get('verdict') == 'ACCEPT']
refused = [r for r in fasterpy_rescored if r.get('verdict') == 'REFUSED_OR_EMPTY']

print(f'Total: {len(fasterpy_rescored)}')
print(f'%OPT (your own evaluator): {100*len(accepted)/len(fasterpy_rescored):.1f}%')
print(f'Refused/empty: {len(refused)} ({100*len(refused)/len(fasterpy_rescored):.1f}%)')

pass1_count = sum(1 for r in fasterpy_rescored if r.get('verdict') == 'ACCEPT' or r.get('reason') == 'insufficient_speedup')
print(f'Pass@1 (functionally correct, your evaluator): {100*pass1_count/len(fasterpy_rescored):.1f}%')

if accepted:
    import statistics
    speedups = [r['speedup'] for r in accepted]
    print(f'Median speedup: {statistics.median(speedups):.3f}x')
    print(f'Mean speedup: {statistics.mean(speedups):.3f}x')